In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import numpy as np
import os

import matplotlib.pyplot as plt
import torchvision
import torchvision.io as torchio
from torchvision.io import ImageReadMode 
from torchvision.transforms import Compose
import torchvision.transforms as transforms
from utils.utils import get_sliding_windows, reconstruct_from_windows

/Users/ggs/miniconda3/envs/degradi/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
"""Generate single mask from PBR tensor"""
RESOLUTION=1024

pbr_tensor = torch.load("images-and-masks/torch-data/data/data_11")
DEVICE="cpu"
resizer = torchvision.transforms.Resize((3041, 3495))
pbr_tensor=resizer(pbr_tensor)
pbr_tensor.shape


torch.Size([7, 3041, 3495])

In [24]:

# Cut image into windows
windows, positions = get_sliding_windows(pbr_tensor, window_size=2*RESOLUTION) # windows = list of tensors of shape (Channels, 2*RESOLUTION, 2*RESOLUTION)
                                                                                    # positions = list of (y, x) tuples
# Move to device and add batch dimension
windows = [torchvision.transforms.Resize((RESOLUTION, RESOLUTION))(window).unsqueeze(0).to(DEVICE) for window in windows] # windows = list of tensors of shape (1, Channels, RESOLUTION, RESOLUTION)

 


In [25]:

# Generate prediction
output = [window for window in windows] # output = list of tensors of shape (1, 1, RESOLUTION, RESOLUTION)
output = [out.squeeze(0).cpu() for out in output] # output = list of tensors of shape (1, RESOLUTION, RESOLUTION)

# Delete batch dimension and resize to original window size
masks = [torchvision.transforms.Resize((2*RESOLUTION, 2*RESOLUTION))(out[0, :].unsqueeze(0)).squeeze(0)  for out in output]  # masks = list of tensors of shape (2*RESOLUTION, 2*RESOLUTION)

# Reconstruct full-size mask
mask = reconstruct_from_windows(masks, positions, original_shape = pbr_tensor.size()[1:], window_size=2*RESOLUTION)  # mask = tensor of shape (H, W)

In [26]:
mask.shape

torch.Size([3041, 3495])